# 7. 통합: 이벤트에서 승인된 결과까지

근거: [설계 글 §2–6](https://x.com/DeepSeekHarness/status/2116857257245987236?s=20). 교육용 축소 구현이며 외부 메시지를 보내지 않습니다.

**실행:** Python (Harness Lab) 커널에서 위에서 아래로 실행하세요. 각 노트북은 새 커널에서 독립 실행됩니다. 기본 셀은 외부 API를 호출하지 않습니다.

## 시나리오
IM의 요청과 GitHub의 별도 요청을 받아 Worker 두 개가 병행 조사합니다. 작업이 끝나기 전 IM 작업의 후속 입력을 받습니다. Worker는 로컬 문서만 읽고 Router가 보고서 초안을 만듭니다. LangGraph interrupt에서 사용자가 승인하거나 거부하면 결과를 **로컬 상태에만** 기록합니다.

이번 통합은 2계층, 구독 pass-through, 실행 격리, 이벤트 수신, LangChain 도구, LangGraph 승인 흐름을 연결합니다. 03의 문맥 회수와 06의 영속 복구는 확장 과제로 남깁니다. 이 데모가 production Super-agent의 모든 성질을 충족한다고 주장하지 않습니다.

In [1]:
import asyncio
from typing import TypedDict
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

@tool
def read_local(topic: str) -> str:
    """주제별 로컬 교육 문서를 읽습니다."""
    return {'routing':'Router는 배정, Worker는 실행을 담당한다.',
            'state':'상태는 compute와 분리해 보존한다.'}[topic]

class Work(TypedDict):
    topic: str
    evidence: str
wb=StateGraph(Work)
wb.add_node('research',lambda s:{'evidence':read_local.invoke({'topic':s['topic']})})
wb.add_edge(START,'research'); wb.add_edge('research',END)
worker=wb.compile()

async def experiment():
    subscriptions={}; inboxes={}; jobs={}; observed=[]
    release=asyncio.Event(); entered=asyncio.Queue(); completions=asyncio.Queue()
    async def run(wid,topic):
        await entered.put(wid)
        await release.wait()
        result=await worker.ainvoke({'topic':topic,'evidence':''})
        await completions.put({'worker':wid,'evidence':result['evidence'],'updates':list(inboxes[wid])})
    def admit(thread,topic,text):
        observed.append((thread,text))
        if thread in subscriptions:
            wid=subscriptions[thread]; inboxes[wid].append(text)
            return wid
        wid=f'w{len(jobs)+1}'; subscriptions[thread]=wid; inboxes[wid]=[text]
        jobs[wid]=asyncio.create_task(run(wid,topic))
        return wid
    try:
        admit('im:42','routing','라우팅 조사')
        admit('github:9','state','상태 조사')
        started={await asyncio.wait_for(entered.get(),2),await asyncio.wait_for(entered.get(),2)}
        assert started=={'w1','w2'}
        assert not any(t.done() for t in jobs.values())
        assert admit('im:42','routing','짧게 보고해 주세요')=='w1'
        assert len(jobs)==2  # 후속 입력은 새 Worker를 만들지 않음
        release.set()
        await asyncio.wait_for(asyncio.gather(*jobs.values()),5)
        results=[await completions.get() for _ in jobs]
        results.sort(key=lambda x:x['worker'])
        assert len(results[0]['updates'])==2 and len(results[1]['updates'])==1
        return results,observed
    finally:
        for t in jobs.values():
            if not t.done(): t.cancel()
        await asyncio.gather(*jobs.values(),return_exceptions=True)
results,observed=await experiment()
print(results)

[{'worker': 'w1', 'evidence': 'Router는 배정, Worker는 실행을 담당한다.', 'updates': ['라우팅 조사', '짧게 보고해 주세요']}, {'worker': 'w2', 'evidence': '상태는 compute와 분리해 보존한다.', 'updates': ['상태 조사']}]


In [2]:
class Publication(TypedDict):
    evidence: list[str]
    draft: str
    delivery: str

def compose(s): return {'draft':'검토 결과\n'+'\n'.join('- '+x for x in s['evidence'])}
def approve(s):
    answer=interrupt({'action':'로컬 보고서 확정','draft':s['draft']})
    return {'delivery':s['draft'] if answer is True else '전달 거부'}
b=StateGraph(Publication)
b.add_node('router_compose',compose); b.add_node('approval',approve)
b.add_edge(START,'router_compose'); b.add_edge('router_compose','approval'); b.add_edge('approval',END)
app=b.compile(checkpointer=InMemorySaver())
initial={'evidence':[r['evidence'] for r in results],'draft':'','delivery':''}
config={'configurable':{'thread_id':'approval-1'}}
paused=app.invoke(initial,config)
assert paused['__interrupt__'] and paused['delivery']==''
accepted=app.invoke(Command(resume=True),config)
assert 'Router' in accepted['delivery']
config2={'configurable':{'thread_id':'approval-2'}}
app.invoke(initial,config2)
rejected=app.invoke(Command(resume=False),config2)
assert rejected['delivery']=='전달 거부'
print(accepted['delivery']); print(rejected['delivery'])

검토 결과
- Router는 배정, Worker는 실행을 담당한다.
- 상태는 compute와 분리해 보존한다.
전달 거부


## 결과 해석과 운영 차이
두 Worker가 동시에 장벽에 도달했으므로 한 Worker의 완료를 기다린 뒤 다음을 시작하지 않았음을 확인했습니다. 이는 LLM 성능 benchmark가 아닙니다. 각 Worker가 받은 updates 목록이 달라 문맥 격리를 확인했습니다. 여기서 후속 지시는 기록만 하며 진행 중 모델 prompt 변경까지 구현하지 않았습니다.

interrupt 노드는 재개 시 처음부터 다시 실행될 수 있습니다. 외부 전송 같은 부작용을 interrupt 앞에 두지 마세요. thread_id는 이 예제에서는 승인 실행의 checkpoint 키입니다. 설계 글의 외부 Thread와 자동으로 동일하지 않습니다. 실제 전송은 승인 후 outbox와 멱등키를 통해 처리해야 합니다.

## 직접 바꿔 보기

합격 기준: (1) read_local에 새 자료 provider를 연결 (2) Worker가 실패해도 Router가 오류를 사건으로 받기 (3) DB를 재연결해 중복 입력을 막기 (4) 승인 거부 시 outbox가 비어 있기. 먼저 한 항목을 골라 assert로 검사하세요.

## 해설과 참고 답안

05의 Protocol/provider 교체를 read_local 내부에 적용할 수 있습니다. Worker 예외는 04의 failed 사건으로 변환하고, 06의 events 기본키로 중복을 거부합니다. DB transaction 안에서 완료 상태와 outbox를 함께 기록하면 저장 후 전송 전 장애를 복구할 수 있습니다. 원격 전송은 수신 측 멱등성도 필요합니다.